<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_12/fl/client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning mit Flower AI — Client-Notebook 🌸

Dieses Notebook führt jede Studierende / jeder Studierende **einmal in Google Colab** aus. Es
verbindet sich als **Flower-Client** mit dem Server der Dozentin / des Dozenten, trainiert ein
neuronales Netz auf einem eigenen Ausschnitt des MNIST-Datensatzes und schickt die gelernten
Gewichte zur Aggregation zurück (Federated Averaging).

## Nur zwei Dinge musst du anpassen

In Abschnitt 3 unten:

1. `SERVER_ADDRESS` — die Adresse, die dir die Dozentin / der Dozent mitteilt.
2. `CLIENT_ID` — deine persönliche, dir zugewiesene Nummer (0–19).

Danach das Notebook einfach von oben nach unten ausführen ("Laufzeit → Alle Zellen ausführen").

## 1. Installation

In [ ]:
!pip install -q "flwr>=1.32" tensorflow matplotlib numpy

## 2. Gemeinsame Hilfsfunktionen (`common.py`)

Identisch zum Server-Notebook: Der Inhalt der Projektdatei `common.py` wird hier materialisiert,
damit dieses Notebook eigenständig in Colab läuft.

In [ ]:
!wget -O common.py https://raw.githubusercontent.com/dgaida/wpf_dlml_th_public/main/assets/exercises/week_12/fl/common.py

## 3. Konfiguration — **hier anpassen**

Trage hier deine individuellen Werte ein. Alles andere im Notebook kann unverändert bleiben.

In [ ]:
# -----------------------------------------------------------------------
# NUR DIESE ZWEI ZEILEN MÜSSEN ANGEPASST WERDEN:
# -----------------------------------------------------------------------
SERVER_ADDRESS = "6.tcp.ngrok.io:21057"  # <-- vom Dozenten/von der Dozentin mitgeteilte Adresse
CLIENT_ID = 0                               # <-- deine individuelle Client-ID (0-19)
# -----------------------------------------------------------------------

# Muss mit der Einstellung im Server-Notebook übereinstimmen ("iid", "shard" oder "dominant").
PARTITION_STRATEGY = "iid"

## 4. Daten laden und partitionieren

Jeder Client erhält reproduzierbar seinen eigenen, disjunkten Ausschnitt von MNIST.

In [ ]:
import common

x_train, y_train = common.load_client_data(CLIENT_ID, strategy=PARTITION_STRATEGY)
x_test, y_test = common.load_test_data()

print(f"Client {CLIENT_ID}: {x_train.shape[0]} lokale Trainingsbilder, Strategie='{PARTITION_STRATEGY}'")

## 5. Optional: Welche Ziffern besitzt mein Client?

In [ ]:
import matplotlib.pyplot as plt

common.plot_distribution(y_train, client_id=CLIENT_ID)
plt.show()

## 6. Modell

Dasselbe Modell wie auf dem Server — nur identische Architekturen lassen sich per FedAvg aggregieren.

In [ ]:
model = common.create_model()
model.summary()

## 7. Flower-Client implementieren

Ein `NumPyClient` muss drei Methoden implementieren:

- **`get_parameters`**: aktuelle lokale Gewichte an den Server zurückgeben.
- **`fit`**: die vom Server erhaltenen globalen Gewichte übernehmen, lokal trainieren (Anzahl
  Epochen kommt über `config["local_epochs"]` vom Server) und die neuen Gewichte zurückgeben.
- **`evaluate`**: die globalen Gewichte auf dem eigenen (oder dem globalen Test-)Datensatz
  evaluieren.

In [ ]:
import flwr as fl
from flwr.common import NDArrays, Scalar


class FlowerClient(fl.client.NumPyClient):
    """Flower-Client für einen einzelnen Studierenden im FL-Demoprojekt.

    Kapselt das lokale Keras-Modell sowie die lokale Trainings- und
    Testpartition dieses Clients und implementiert die von Flower
    erwartete :class:`flwr.client.NumPyClient`-Schnittstelle.

    Attributes:
        model: Das lokale Keras-Modell (identische Architektur wie der Server).
        x_train: Lokale Trainingsbilder dieses Clients.
        y_train: Lokale Trainingslabels dieses Clients.
        x_test: Globaler (nicht partitionierter) Testdatensatz.
        y_test: Globale Testlabels.
    """

    def __init__(self, model, x_train, y_train, x_test, y_test) -> None:
        """Initialisiert den Client mit Modell und Datenpartition.

        Args:
            model: Das zu trainierende Keras-Modell.
            x_train: Lokale Trainingsbilder.
            y_train: Lokale Trainingslabels.
            x_test: Globale Testbilder für die Evaluation.
            y_test: Globale Testlabels für die Evaluation.
        """
        self.model = model
        self.x_train = x_train
        self.y_train = y_train
        self.x_test = x_test
        self.y_test = y_test

    def get_parameters(self, config: dict[str, Scalar]) -> NDArrays:
        """Gibt die aktuellen lokalen Modellgewichte zurück.

        Args:
            config: Vom Server übergebene Konfiguration (hier ungenutzt).

        Returns:
            NDArrays: Liste der aktuellen Gewichts-Arrays des lokalen Modells.
        """
        return self.model.get_weights()

    def fit(
        self, parameters: NDArrays, config: dict[str, Scalar]
    ) -> tuple[NDArrays, int, dict[str, Scalar]]:
        """Trainiert das Modell lokal ausgehend von den globalen Parametern.

        Args:
            parameters: Aktuelle globale Modellgewichte vom Server.
            config: Trainingskonfiguration vom Server, insbesondere
                ``local_epochs`` (Anzahl lokaler Trainings-Epochen).

        Returns:
            tuple: ``(neue_gewichte, anzahl_beispiele, metriken)`` gemäß der
            von Flower erwarteten Rückgabestruktur.
        """
        self.model.set_weights(parameters)
        local_epochs = int(config.get("local_epochs", 1))
        history = self.model.fit(
            self.x_train,
            self.y_train,
            epochs=local_epochs,
            batch_size=common.FLConfig().batch_size,
            verbose=0,
        )
        train_accuracy = float(history.history["accuracy"][-1])
        print(
            f"[Client {CLIENT_ID}] Runde {config.get('server_round', '?')}: "
            f"lokal trainiert ({local_epochs} Epoche(n), "
            f"train_accuracy={train_accuracy:.4f})"
        )
        return self.model.get_weights(), len(self.x_train), {"accuracy": train_accuracy}

    def evaluate(
        self, parameters: NDArrays, config: dict[str, Scalar]
    ) -> tuple[float, int, dict[str, Scalar]]:
        """Evaluiert die globalen Parameter auf dem lokalen Testdatensatz.

        Args:
            parameters: Aktuelle globale Modellgewichte vom Server.
            config: Evaluationskonfiguration vom Server (hier ungenutzt).

        Returns:
            tuple: ``(loss, anzahl_beispiele, metriken)`` gemäß der von
            Flower erwarteten Rückgabestruktur.
        """
        self.model.set_weights(parameters)
        loss, accuracy = self.model.evaluate(self.x_test, self.y_test, verbose=0)
        print(f"[Client {CLIENT_ID}] Lokale Evaluation: loss={loss:.4f}, accuracy={accuracy:.4f}")
        return float(loss), len(self.x_test), {"accuracy": float(accuracy)}


## 8. Verbindung zum Server herstellen

Diese Zelle **blockiert**, bis der Server alle Kommunikationsrunden abgeschlossen hat (oder die
Verbindung endet). Solange läuft im Hintergrund abwechselnd lokales Training und Evaluation.

> Auch hier ist die Meldung `Using start_client() is deprecated` erwartet und unproblematisch
> (siehe Hinweis im Server-Notebook).

In [ ]:
flower_client = FlowerClient(model, x_train, y_train, x_test, y_test)

print(f"Client {CLIENT_ID}: verbinde mit {SERVER_ADDRESS} ...")
fl.client.start_client(
    server_address=SERVER_ADDRESS,
    client=flower_client.to_client(),
)
print(f"Client {CLIENT_ID}: Training beendet, Verbindung geschlossen.")


## Fertig! 🎉

Sobald diese Zelle durchgelaufen ist, hat dein Client an allen Runden teilgenommen, für die der
Server aktiv war. Die Konsolenausgaben oben zeigen deine lokale Trainings- und Evaluations-Accuracy
pro Runde.

**Optional:** Ändere `PARTITION_STRATEGY` auf `"shard"` oder `"dominant"` (nach Absprache mit der
Dozentin / dem Dozenten, da alle Clients dieselbe Strategie verwenden müssen!) und beobachte, wie
sich eine Non-IID-Datenverteilung auf deine lokale Klassenverteilung (Abschnitt 5) und die
Trainingsergebnisse auswirkt.